# Limpieza del núcleo — IPV

Primera transformación del **Índice de Precios de Vivienda (IPV)**, base 2025. Se conserva el archivo oficial sin modificar y se prepara una tabla en formato largo para SQL.

**Nota de ejecución:** las celdas están ordenadas para ejecutarse de arriba abajo. La normalización territorial también puede repetirse sin perder los códigos.

## 1. Lectura del archivo original

Se usa `header=None` para que pandas no confunda las filas informativas con la cabecera de la tabla.

In [1]:
import pandas as pd
from pathlib import Path

ruta = Path("../data/raw/core/base25_raw.xlsx")

df_ipv = pd.read_excel(ruta, header=None)

display(df_ipv.iloc[:14, :12])

c:\Users\20pab\anaconda3\envs\data-analysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7,8,9,10,11
0,Índice de Precios de Vivienda. Base 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Índice de Precios de Vivienda (IPV). Base 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Índices por CCAA: general, vivienda nueva y de...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"Unidades: Índice, Tasas",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,,Índice,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,,2026T2,2026T1,2025T4,2025T3,2025T2,2025T1,2024T4,2024T3,2024T2,2024T1,2023T4
8,Nacional,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,General,111.095,107.458,103.805,101.937,99.043,95.215,91.956,90.337,87.87,84.833,82.649


## 2. Reconocimiento de la estructura original

La fila de índice 6 identifica cuatro bloques de medidas y la fila 7 sus trimestres. Cada territorio ocupa cuatro filas: etiqueta territorial, índice general, vivienda nueva y vivienda de segunda mano.

In [2]:
display(df_ipv.iloc[6:9, :25])

,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
6,,Índice,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,,2026T2,2026T1,2025T4,2025T3,2025T2,2025T1,2024T4,2024T3,2024T2,...,2022T4,2022T3,2022T2,2022T1,2021T4,2021T3,2021T2,2021T1,2020T4,2020T3
8,Nacional,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
print(df_ipv.iloc[6].dropna())

0                                   
1                             Índice
79              Variación trimestral
157                  Variación anual
235    Variación en lo que va de año
Name: 6, dtype: object


In [4]:
display(df_ipv.iloc[8:20, 0:4])

,0,1,2,3
8,Nacional,NaN,NaN,NaN
9,General,111.095,107.458,103.805
10,Vivienda nueva,107.228,106.129,102.559
11,Vivienda segunda mano,111.682,107.666,104.001
12,01 Andalucía,NaN,NaN,NaN
13,General,110.903,108.061,103.77
14,Vivienda nueva,108.261,107.827,103.153
15,Vivienda segunda mano,111.393,108.09,103.873
16,02 Aragón,NaN,NaN,NaN
17,General,113.472,109.195,103.98


## 3. Selección del bloque de índices y etiquetas

Se usan las columnas 1–78 (índices) y la 0 (etiquetas). Los otros bloques son tasas de variación y no se necesitan en esta primera tabla. Las filas de datos territoriales van de la posición 8 a la 87; las notas finales quedan fuera.

In [5]:
# Trabajar desde df_ipv garantiza que esta celda se pueda volver a ejecutar sin acumular cambios.
ipv_indices = df_ipv.iloc[8:88, :79].copy()
periodos = df_ipv.iloc[7, 1:79].tolist()

ipv_indices["territorio"] = ipv_indices.iloc[:, 0].where(
    ipv_indices.index % 4 == 0
).ffill()

ipv_indices = ipv_indices[ipv_indices.index % 4 != 0].copy()

ipv_indices = ipv_indices.rename(columns={0: "tipo_vivienda"})
ipv_indices.columns = ["tipo_vivienda"] + periodos + ["territorio"]
display(ipv_indices.iloc[:9, :6])

,tipo_vivienda,2026T2,2026T1,2025T4,2025T3,2025T2
9,General,111.095,107.458,103.805,101.937,99.043
10,Vivienda nueva,107.228,106.129,102.559,100.371,99.81
11,Vivienda segunda mano,111.682,107.666,104.001,102.183,98.923
13,General,110.903,108.061,103.77,101.791,99.074
14,Vivienda nueva,108.261,107.827,103.153,100.656,99.336
15,Vivienda segunda mano,111.393,108.09,103.873,101.981,99.03
17,General,113.472,109.195,103.98,102.545,98.984
18,Vivienda nueva,108.929,108.18,101.638,102.213,99.221
19,Vivienda segunda mano,114.367,109.411,104.45,102.611,98.936


## 4. Conversión a formato largo

Cada fila pasará a representar un territorio, un tipo de vivienda y un trimestre. `melt` mantiene las columnas identificadoras y reúne los 78 periodos en una sola columna.

In [6]:
ipv_largo = ipv_indices.melt(
    id_vars=["territorio", "tipo_vivienda"],
    var_name="periodo",
    value_name="indice_ipv"
)
ipv_largo["tipo_vivienda"] = ipv_largo["tipo_vivienda"].str.strip()

display(ipv_largo.head(10))
print(ipv_largo.shape)

,territorio,tipo_vivienda,periodo,indice_ipv
0,Nacional,General,2026T2,111.095
1,Nacional,Vivienda nueva,2026T2,107.228
2,Nacional,Vivienda segunda mano,2026T2,111.682
3,01 Andalucía,General,2026T2,110.903
4,01 Andalucía,Vivienda nueva,2026T2,108.261
5,01 Andalucía,Vivienda segunda mano,2026T2,111.393
6,02 Aragón,General,2026T2,113.472
7,02 Aragón,Vivienda nueva,2026T2,108.929
8,02 Aragón,Vivienda segunda mano,2026T2,114.367
9,"03 Asturias, Principado de",General,2026T2,114.014


(4680, 4)


## 5. Tipos de datos y comprobaciones iniciales

Se separa el año y el trimestre; los índices se convierten a valores numéricos. La conversión no numérica a `NaN` se analiza antes de decidir si procede eliminar alguna observación.

In [7]:
ipv_largo["anio"] = ipv_largo["periodo"].str[:4].astype(int)
ipv_largo["trimestre"] = ipv_largo["periodo"].str[-1].astype(int)

ipv_largo["indice_ipv"] = pd.to_numeric(
    ipv_largo["indice_ipv"], errors="coerce"
)

print("Dimensiones:", ipv_largo.shape)
print("Territorios:", ipv_largo["territorio"].nunique())
print("Tipos de vivienda:", ipv_largo["tipo_vivienda"].unique())
print("Periodos:", ipv_largo["periodo"].nunique())
print("Índices nulos:", ipv_largo["indice_ipv"].isna().sum())

print("Duplicados:", ipv_largo.duplicated(
    subset=["territorio", "tipo_vivienda", "periodo"]
).sum())

Dimensiones: (4680, 6)
Territorios: 20
Tipos de vivienda: <ArrowStringArray>
['General', 'Vivienda nueva', 'Vivienda segunda mano']
Length: 3, dtype: str
Periodos: 78
Índices nulos: 312
Duplicados: 0


## 6. Valores ausentes

En los resultados observados, los **312 nulos** corresponden exclusivamente a las series de vivienda nueva y segunda mano de Ceuta y Melilla (2 territorios × 2 tipos × 78 trimestres). Se conservan como datos no disponibles; no se imputan.

In [8]:
nulos = ipv_largo[ipv_largo["indice_ipv"].isna()]

display(nulos.head(15))

,territorio,tipo_vivienda,periodo,indice_ipv,anio,trimestre
55,18 Ceuta,Vivienda nueva,2026T2,NaN,2026,2
56,18 Ceuta,Vivienda segunda mano,2026T2,NaN,2026,2
58,19 Melilla,Vivienda nueva,2026T2,NaN,2026,2
59,19 Melilla,Vivienda segunda mano,2026T2,NaN,2026,2
115,18 Ceuta,Vivienda nueva,2026T1,NaN,2026,1
116,18 Ceuta,Vivienda segunda mano,2026T1,NaN,2026,1
118,19 Melilla,Vivienda nueva,2026T1,NaN,2026,1
119,19 Melilla,Vivienda segunda mano,2026T1,NaN,2026,1
175,18 Ceuta,Vivienda nueva,2025T4,NaN,2025,4
176,18 Ceuta,Vivienda segunda mano,2025T4,NaN,2025,4


In [9]:
print("Nulos por territorio:")
print(nulos["territorio"].value_counts())

print("\nNulos por tipo de vivienda:")
print(nulos["tipo_vivienda"].value_counts())

print("\nNulos por periodo:")
print(nulos["periodo"].value_counts().sort_index())

Nulos por territorio:
territorio
18 Ceuta      156
19 Melilla    156
Name: count, dtype: int64

Nulos por tipo de vivienda:
tipo_vivienda
Vivienda nueva           156
Vivienda segunda mano    156
Name: count, dtype: int64

Nulos por periodo:
periodo
2007T1    4
2007T2    4
2007T3    4
2007T4    4
2008T1    4
         ..
2025T2    4
2025T3    4
2025T4    4
2026T1    4
2026T2    4
Name: count, Length: 78, dtype: int64


## 7. Normalización territorial

En el Excel, los nombres de comunidad empiezan por un código (`03 Asturias, Principado de`), salvo `Nacional`. **Problema corregido:** extraer el código desde una columna que previamente se había limpiado producía códigos nulos al repetir la celda.

La solución es construir siempre el mapa de códigos desde `ipv_indices`, que conserva los nombres originales. Así esta celda se puede ejecutar varias veces y el resultado no cambia.

In [10]:
print(ipv_largo["territorio"].unique())

<ArrowStringArray>
[                      'Nacional',                   '01 Andalucía',
                      '02 Aragón',     '03 Asturias, Principado de',
              '04 Balears, Illes',                    '05 Canarias',
                   '06 Cantabria',             '07 Castilla y León',
        '08 Castilla - La Mancha',                    '09 Cataluña',
        '10 Comunitat Valenciana',                 '11 Extremadura',
                     '12 Galicia',        '13 Madrid, Comunidad de',
           '14 Murcia, Región de', '15 Navarra, Comunidad Foral de',
                  '16 País Vasco',                   '17 Rioja, La',
                       '18 Ceuta',                     '19 Melilla']
Length: 20, dtype: str


In [11]:
# Construir el catálogo desde las etiquetas originales, no desde ipv_largo modificado.
territorios_originales = ipv_indices["territorio"].drop_duplicates()
catalogo = pd.DataFrame({"territorio_original": territorios_originales})
catalogo["cod_territorio"] = catalogo["territorio_original"].str.extract(r"^(\d{2})", expand=False)
catalogo["territorio"] = catalogo["territorio_original"].str.replace(
    r"^\d{2}\s+", "", regex=True
)

# Crear un mapa nombre limpio -> código a partir de la fuente original.
mapa_codigos = catalogo.set_index("territorio")["cod_territorio"]

# Quitar el prefijo del nombre (si lo tiene) y asignar código usando el mapa.
# Estas dos operaciones son seguras al repetir esta celda.
ipv_largo["territorio"] = ipv_largo["territorio"].str.replace(
    r"^\d{2}\s+", "", regex=True
)
ipv_largo["cod_territorio"] = ipv_largo["territorio"].map(mapa_codigos)

display(ipv_largo[["cod_territorio", "territorio"]].drop_duplicates())

# Validar: solo el agregado Nacional debe tener código ausente.
sin_codigo = ipv_largo.loc[ipv_largo["cod_territorio"].isna(), "territorio"].unique().tolist()
assert sin_codigo == ["Nacional"], f"Territorios sin código inesperados: {sin_codigo}"
print("Códigos ausentes (solo Nacional):", ipv_largo["cod_territorio"].isna().sum())

,cod_territorio,territorio
0,NaN,Nacional
3,01,Andalucía
6,02,Aragón
9,03,"Asturias, Principado de"
12,04,"Balears, Illes"
15,05,Canarias
18,06,Cantabria
21,07,Castilla y León
24,08,Castilla - La Mancha
27,09,Cataluña


Códigos ausentes (solo Nacional): 234


## 8. Ordenación y validación final

Se mantiene una única observación por territorio, trimestre y tipo de vivienda. Antes de exportar se comprueban dimensiones, duplicados, nulos y códigos territoriales.

In [12]:
ipv_largo = ipv_largo[
    [
        "cod_territorio",
        "territorio",
        "anio",
        "trimestre",
        "tipo_vivienda",
        "indice_ipv"
    ]
].copy()

ipv_largo = ipv_largo.sort_values(
    by=["cod_territorio", "anio", "trimestre", "tipo_vivienda"],
    na_position="first"
).reset_index(drop=True)

display(ipv_largo.head(10))
display(ipv_largo.tail(10))

,cod_territorio,territorio,anio,trimestre,tipo_vivienda,indice_ipv
0,NaN,Nacional,2007,1,General,81.118
1,NaN,Nacional,2007,1,Vivienda nueva,61.011
2,NaN,Nacional,2007,1,Vivienda segunda mano,93.482
3,NaN,Nacional,2007,2,General,83.435
4,NaN,Nacional,2007,2,Vivienda nueva,62.848
5,NaN,Nacional,2007,2,Vivienda segunda mano,96.035
6,NaN,Nacional,2007,3,General,84.333
7,NaN,Nacional,2007,3,Vivienda nueva,64.169
8,NaN,Nacional,2007,3,Vivienda segunda mano,96.269
9,NaN,Nacional,2007,4,General,83.733


,cod_territorio,territorio,anio,trimestre,tipo_vivienda,indice_ipv
4670,19,Melilla,2025,3,Vivienda segunda mano,NaN
4671,19,Melilla,2025,4,General,102.650
4672,19,Melilla,2025,4,Vivienda nueva,NaN
4673,19,Melilla,2025,4,Vivienda segunda mano,NaN
4674,19,Melilla,2026,1,General,108.257
4675,19,Melilla,2026,1,Vivienda nueva,NaN
4676,19,Melilla,2026,1,Vivienda segunda mano,NaN
4677,19,Melilla,2026,2,General,113.245
4678,19,Melilla,2026,2,Vivienda nueva,NaN
4679,19,Melilla,2026,2,Vivienda segunda mano,NaN


In [13]:
# La combinación de columnas siguiente identifica cada observación.
clave = ["territorio", "anio", "trimestre", "tipo_vivienda"]

assert len(ipv_largo) == 4680, "Número inesperado de observaciones"
assert ipv_largo["territorio"].nunique() == 20, "Número inesperado de territorios"
assert ipv_largo["tipo_vivienda"].nunique() == 3, "Número inesperado de tipos de vivienda"
assert ipv_largo.duplicated(subset=clave).sum() == 0, "Se han detectado duplicados"
assert ipv_largo["cod_territorio"].isna().sum() == 234, "Códigos territoriales ausentes inesperados"
assert ipv_largo["indice_ipv"].isna().sum() == 312, "Revisar la distribución de índices nulos"

print("Validaciones superadas")
print("Observaciones:", len(ipv_largo))
print("Nulos en indice_ipv:", ipv_largo["indice_ipv"].isna().sum())
print("Duplicados:", ipv_largo.duplicated(subset=clave).sum())

Validaciones superadas
Observaciones: 4680
Nulos en indice_ipv: 312
Duplicados: 0


## Resultado de esta etapa

El DataFrame `ipv_largo` queda preparado con seis columnas: `cod_territorio`, `territorio`, `anio`, `trimestre`, `tipo_vivienda` e `indice_ipv`. Los valores ausentes de Ceuta y Melilla siguen presentes y documentados.

**No se exporta todavía:** el siguiente paso, cuando se decida, será contrastar algunos valores contra el Excel original y escribir el CSV procesado.

In [14]:
assert len(ipv_largo) == 4680
assert ipv_largo["territorio"].nunique() == 20
assert ipv_largo["tipo_vivienda"].nunique() == 3
assert ipv_largo["indice_ipv"].isna().sum() == 312

assert ipv_largo.duplicated(
    subset=["cod_territorio", "anio", "trimestre", "tipo_vivienda"]
).sum() == 0

assert ipv_largo.loc[
    ipv_largo["territorio"] != "Nacional",
    "cod_territorio"
].notna().all()

In [15]:
ruta_salida = Path("../data/processed/core/ipv.csv")

ruta_salida.parent.mkdir(parents=True, exist_ok=True)

ipv_largo.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

In [16]:
ipv_exportado = pd.read_csv(
    ruta_salida,
    dtype={"cod_territorio": "string"}
)

print(ipv_exportado.shape)
display(ipv_exportado.head())

(4680, 6)


,cod_territorio,territorio,anio,trimestre,tipo_vivienda,indice_ipv
0,<NA>,Nacional,2007,1,General,81.118
1,<NA>,Nacional,2007,1,Vivienda nueva,61.011
2,<NA>,Nacional,2007,1,Vivienda segunda mano,93.482
3,<NA>,Nacional,2007,2,General,83.435
4,<NA>,Nacional,2007,2,Vivienda nueva,62.848


## IPC

In [18]:
# IPC: carga del archivo original
ruta_ipc = Path("../data/raw/core/ipcgrupos_raw.xlsx")

df_ipc = pd.read_excel(ruta_ipc, header=None)

print(df_ipc.iloc[6].dropna())

c:\Users\20pab\anaconda3\envs\data-analysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


0                                   
1                             Índice
298                Variación mensual
595                  Variación anual
892    Variación en lo que va de año
Name: 6, dtype: object


In [19]:
display(df_ipc.iloc[8:39, 0:4])

,0,1,2,3
8,Nacional,NaN,NaN,NaN
9,Índice general,104.978,104.638,103.899
10,01 Alimentos y bebidas no alcohólicas,101.934,101.71,101.843
11,02 Bebidas alcohólicas y tabaco,103.289,103.106,103.166
12,03 Vestido y calzado,98.092,91.743,92.649
13,"04 Vivienda, agua, electricidad, gas y otr...",107.637,106.491,105.803
14,"05 Muebles, artículos del hogar y artículo...",101.45,101.245,101.264
15,06 Sanidad,102.357,102.297,102.166
16,07 Transporte,110.574,110.33,106.766
17,08 Información y comunicaciones,102.338,102.348,101.244


In [21]:
ipc_indices = df_ipc.iloc[8:308, :298].copy()
periodos_ipc = df_ipc.iloc[7, 1:298].tolist()

ipc_indices["territorio"] = ipc_indices.iloc[:, 0].where(
    (ipc_indices.index - 8) % 15 == 0
).ffill()

ipc_indices = ipc_indices[
    (ipc_indices.index - 8) % 15 != 0
].copy()

ipc_indices.columns = ["grupo_consumo"] + periodos_ipc + ["territorio"]

display(ipc_indices.iloc[:16, :6])

print("Dimensiones:", ipc_indices.shape)
print("Territorios:", ipc_indices["territorio"].nunique())
print("Grupos de consumo:", ipc_indices["grupo_consumo"].nunique())

,grupo_consumo,2026M09,2026M08,2026M07,2026M06,2026M05
9,Índice general,104.978,104.638,103.899,103.598,102.951
10,01 Alimentos y bebidas no alcohólicas,101.934,101.71,101.843,102.527,102.441
11,02 Bebidas alcohólicas y tabaco,103.289,103.106,103.166,103.399,103.489
12,03 Vestido y calzado,98.092,91.743,92.649,103.194,104.064
13,"04 Vivienda, agua, electricidad, gas y otr...",107.637,106.491,105.803,104.168,99.695
14,"05 Muebles, artículos del hogar y artículo...",101.45,101.245,101.264,101.563,101.346
15,06 Sanidad,102.357,102.297,102.166,102.211,102.051
16,07 Transporte,110.574,110.33,106.766,104.338,105.723
17,08 Información y comunicaciones,102.338,102.348,101.244,100.856,100.862
18,"09 Actividades recreativas, deporte y cultura",103.541,106.198,105.432,103.102,101.166


Dimensiones: (280, 299)
Territorios: 20
Grupos de consumo: 14


In [22]:
ipc_largo = ipc_indices.melt(
    id_vars=["territorio", "grupo_consumo"],
    var_name="periodo",
    value_name="indice_ipc"
)

ipc_largo["grupo_consumo"] = ipc_largo["grupo_consumo"].str.strip()

ipc_largo["anio"] = ipc_largo["periodo"].str[:4].astype(int)
ipc_largo["mes"] = ipc_largo["periodo"].str[-2:].astype(int)

ipc_largo["indice_ipc"] = pd.to_numeric(
    ipc_largo["indice_ipc"], errors="coerce"
)

print("Dimensiones:", ipc_largo.shape)
print("Territorios:", ipc_largo["territorio"].nunique())
print("Grupos de consumo:", ipc_largo["grupo_consumo"].nunique())
print("Periodos:", ipc_largo["periodo"].nunique())
print("Índices nulos:", ipc_largo["indice_ipc"].isna().sum())

print("Duplicados:", ipc_largo.duplicated(
    subset=["territorio", "grupo_consumo", "periodo"]
).sum())

Dimensiones: (83160, 6)
Territorios: 20
Grupos de consumo: 14
Periodos: 297
Índices nulos: 0
Duplicados: 0


In [23]:
ipc_largo["cod_territorio"] = ipc_largo["territorio"].str.extract(r"^(\d{2})")

ipc_largo["territorio"] = ipc_largo["territorio"].str.replace(
    r"^\d{2}\s+", "", regex=True
)

display(
    ipc_largo[["cod_territorio", "territorio"]].drop_duplicates()
)

print("Códigos nulos:", ipc_largo["cod_territorio"].isna().sum())

,cod_territorio,territorio
0,NaN,Nacional
14,01,Andalucía
28,02,Aragón
42,03,"Asturias, Principado de"
56,04,"Balears, Illes"
70,05,Canarias
84,06,Cantabria
98,07,Castilla y León
112,08,Castilla - La Mancha
126,09,Cataluña


Códigos nulos: 4158


In [26]:
ipc_largo = ipc_largo[
    [
        "cod_territorio",
        "territorio",
        "anio",
        "mes",
        "grupo_consumo",
        "indice_ipc"
    ]
].copy()

ipc_largo = ipc_largo.sort_values(
    by=["cod_territorio", "anio", "mes", "grupo_consumo"],
    na_position="first"
).reset_index(drop=True)

assert len(ipc_largo) == 83160
assert ipc_largo["territorio"].nunique() == 20
assert ipc_largo["grupo_consumo"].nunique() == 14
assert ipc_largo["indice_ipc"].isna().sum() == 0

assert ipc_largo.duplicated(
    subset=["cod_territorio", "anio", "mes", "grupo_consumo"]
).sum() == 0

assert ipc_largo.loc[
    ipc_largo["territorio"] != "Nacional",
    "cod_territorio"
].notna().all()

print("Validaciones superadas")


Validaciones superadas


In [27]:
ruta_salida = Path("../data/processed/core/ipc.csv")

ruta_salida.parent.mkdir(parents=True, exist_ok=True)

ipc_largo.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

In [29]:
ipc_exportado = pd.read_csv(
    ruta_salida,
    dtype={"cod_territorio": "string"}
)

print(ipc_exportado.shape)
display(ipc_exportado.head())

(83160, 6)


,cod_territorio,territorio,anio,mes,grupo_consumo,indice_ipc
0,<NA>,Nacional,2002,1,01 Alimentos y bebidas no alcohólicas,49.796
1,<NA>,Nacional,2002,1,02 Bebidas alcohólicas y tabaco,38.396
2,<NA>,Nacional,2002,1,03 Vestido y calzado,77.799
3,<NA>,Nacional,2002,1,"04 Vivienda, agua, electricidad, gas y otros c...",51.962
4,<NA>,Nacional,2002,1,"05 Muebles, artículos del hogar y artículos pa...",71.792


## Salarios

In [30]:
df_salarios = pd.read_excel(
    "../data/raw/core/salarios_raw.xlsx",
    header=None
)

display(df_salarios.iloc[:16, :8])

c:\Users\20pab\anaconda3\envs\data-analysis\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,0,1,2,3,4,5,6,7
0,Resultados nacionales y por comunidades aútonomas,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ganancia media anual por trabajador,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sexo y sectores de actividad,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Unidades: €,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,,Total Nacional,Andalucía,Aragón,"Asturias, Principado de","Balears, Illes",Canarias,Cantabria
7,Todos los sectores,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Ambos sexos,29540.26,26089.7,28061.94,28562.31,29075.06,25120.38,27097.17


In [31]:
territorios_salarios = df_salarios.iloc[6, 1:].tolist()

anios_salarios = df_salarios.iloc[8:42:2, 0].tolist()

valores_salarios = df_salarios.iloc[9:43:2, 1:].copy()

print("Territorios:", len(territorios_salarios))
print("Años:", anios_salarios)
print("Dimensiones:", valores_salarios.shape)

display(valores_salarios.head())

Territorios: 18
Años: ['    2024', '    2023', '    2022', '    2021', '    2020', '    2019', '    2018', '    2017', '    2016', '    2015', '    2014', '    2013', '    2012', '    2011', '    2010', '    2009', '    2008']
Dimensiones: (17, 18)


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
9,29540.26,26089.7,28061.94,28562.31,29075.06,25120.38,27097.17,26177.28,26062.05,31730.05,26816.98,24979.44,26547.27,34410.01,26349.36,32605.28,35170.28,27855.4
11,28049.94,25051.51,26822.32,27916.6,27537,24033.57,26568.7,25227.03,24885.63,29978.69,25632.37,23684.22,25279.15,32219.6,25329.87,31199.66,33504.92,26319.06
13,26948.87,24094.32,26012.71,26745.93,27145.79,23096.92,25291.14,24186.89,23751.71,28774.55,24510.34,21922.73,24169.68,31230.73,23851.69,29189.52,32313.73,24903.38
15,25896.82,23154.99,25011.55,25046.6,24134.73,22466.23,24470.7,23564.33,23234.55,28145.02,23448.29,21393.01,23305.86,29512.58,23281.39,28459.37,31063.68,23716.88
17,25165.51,22323.85,24113.57,24527.96,23881.42,21631.17,23495.12,22948.76,22575.01,27100.11,22856.7,20479.3,22834.66,28829.06,22919.62,27995.96,30224.16,23292.63


In [32]:
salarios = valores_salarios.copy()

salarios.columns = territorios_salarios
salarios["anio"] = [int(anio.strip()) for anio in anios_salarios]

salarios_largo = salarios.melt(
    id_vars="anio",
    var_name="territorio",
    value_name="salario_medio"
)

print("Dimensiones:", salarios_largo.shape)
print("Territorios:", salarios_largo["territorio"].nunique())
print("Años:", salarios_largo["anio"].nunique())
print("Tipos de datos:")
print(salarios_largo.dtypes)

Dimensiones: (306, 3)
Territorios: 18
Años: 17
Tipos de datos:
anio              int64
territorio          str
salario_medio    object
dtype: object


In [33]:
valores_numericos = pd.to_numeric(
    salarios_largo["salario_medio"],
    errors="coerce"
)

valores_especiales = salarios_largo[
    valores_numericos.isna()
    & salarios_largo["salario_medio"].notna()
]

print("Valores no numéricos:", len(valores_especiales))

display(
    valores_especiales[
        ["anio", "territorio", "salario_medio"]
    ].drop_duplicates()
)

Valores no numéricos: 0


,anio,territorio,salario_medio


In [34]:
salarios_largo["salario_medio"] = pd.to_numeric(
    salarios_largo["salario_medio"],
    errors="coerce"
)

print("Salarios nulos:", salarios_largo["salario_medio"].isna().sum())

display(
    salarios_largo[
        salarios_largo["salario_medio"].isna()
    ]
)

Salarios nulos: 0


,anio,territorio,salario_medio


In [35]:
salarios_largo["territorio"] = salarios_largo["territorio"].str.strip()

salarios_largo["territorio"] = salarios_largo["territorio"].replace({
    "Total Nacional": "Nacional"
})

print(salarios_largo["territorio"].unique())

<ArrowStringArray>
[                   'Nacional',                   'Andalucía',
                      'Aragón',     'Asturias, Principado de',
              'Balears, Illes',                    'Canarias',
                   'Cantabria',             'Castilla y León',
        'Castilla - La Mancha',                    'Cataluña',
        'Comunitat Valenciana',                 'Extremadura',
                     'Galicia',        'Madrid, Comunidad de',
           'Murcia, Región de', 'Navarra, Comunidad Foral de',
                  'País Vasco',                   'Rioja, La']
Length: 18, dtype: str


In [36]:
mapa_territorios = (
    ipc_largo[["territorio", "cod_territorio"]]
    .drop_duplicates()
    .set_index("territorio")["cod_territorio"]
)

salarios_largo["cod_territorio"] = salarios_largo["territorio"].map(
    mapa_territorios
)

salarios_largo = salarios_largo[
    ["cod_territorio", "territorio", "anio", "salario_medio"]
].copy()

salarios_largo = salarios_largo.sort_values(
    ["cod_territorio", "anio"],
    na_position="first"
).reset_index(drop=True)

print("Territorios:", salarios_largo["territorio"].nunique())
print("Códigos nulos:", salarios_largo["cod_territorio"].isna().sum())

display(salarios_largo.head())

Territorios: 18
Códigos nulos: 17


,cod_territorio,territorio,anio,salario_medio
0,NaN,Nacional,2008,21883.42
1,NaN,Nacional,2009,22511.47
2,NaN,Nacional,2010,22790.20
3,NaN,Nacional,2011,22899.35
4,NaN,Nacional,2012,22726.44


In [37]:
assert len(salarios_largo) == 306
assert salarios_largo["territorio"].nunique() == 18
assert salarios_largo["anio"].nunique() == 17
assert salarios_largo["salario_medio"].isna().sum() == 0

assert salarios_largo.duplicated(
    subset=["cod_territorio", "anio"]
).sum() == 0

assert salarios_largo.loc[
    salarios_largo["territorio"] != "Nacional",
    "cod_territorio"
].notna().all()

assert salarios_largo["anio"].between(2008, 2024).all()
assert (salarios_largo["salario_medio"] > 0).all()

print("Validaciones superadas")

Validaciones superadas


In [38]:
ruta_salida = Path("../data/processed/core/salarios.csv")

ruta_salida.parent.mkdir(parents=True, exist_ok=True)

salarios_largo.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

salarios_exportado = pd.read_csv(
    ruta_salida,
    dtype={"cod_territorio": "string"}
)

print("Dimensiones:", salarios_exportado.shape)
display(salarios_exportado.head())

Dimensiones: (306, 4)


,cod_territorio,territorio,anio,salario_medio
0,<NA>,Nacional,2008,21883.42
1,<NA>,Nacional,2009,22511.47
2,<NA>,Nacional,2010,22790.20
3,<NA>,Nacional,2011,22899.35
4,<NA>,Nacional,2012,22726.44


## Tasado m2

In [39]:
ruta_tasa = Path("../data/raw/core/tasa_raw.csv")

with open(ruta_tasa, "r", encoding="utf-8-sig") as archivo:
    for _ in range(8):
        print(repr(archivo.readline()))



'Año;Trimestre;Valor;Régimen;CPRO;Provincia;Comunidad_Autónoma;CODAUTO\n'
'1995;1;;Libre;null;Ceuta y Melilla;Ceuta y Melilla;0\n'
'1995;2;;Libre;null;Ceuta y Melilla;Ceuta y Melilla;0\n'
'1995;3;;Libre;null;Ceuta y Melilla;Ceuta y Melilla;0\n'
'1995;4;;Libre;null;Ceuta y Melilla;Ceuta y Melilla;0\n'
'1996;1;;Libre;null;Ceuta y Melilla;Ceuta y Melilla;0\n'
'1996;2;;Libre;null;Ceuta y Melilla;Ceuta y Melilla;0\n'
'1996;3;;Libre;null;Ceuta y Melilla;Ceuta y Melilla;0\n'


In [40]:
df_tasa = pd.read_csv(
    ruta_tasa,
    sep=";",
    encoding="utf-8-sig"
)

print("Dimensiones:", df_tasa.shape)
print("\nTipos de datos:")
print(df_tasa.dtypes)

display(df_tasa.head())

Dimensiones: (13711, 8)

Tipos de datos:
Año                     int64
Trimestre               int64
Valor                 float64
Régimen                   str
CPRO                  float64
Provincia                 str
Comunidad_Autónoma        str
CODAUTO               float64
dtype: object


,Año,Trimestre,Valor,Régimen,CPRO,Provincia,Comunidad_Autónoma,CODAUTO
0,1995,1,NaN,Libre,NaN,Ceuta y Melilla,Ceuta y Melilla,0.0
1,1995,2,NaN,Libre,NaN,Ceuta y Melilla,Ceuta y Melilla,0.0
2,1995,3,NaN,Libre,NaN,Ceuta y Melilla,Ceuta y Melilla,0.0
3,1995,4,NaN,Libre,NaN,Ceuta y Melilla,Ceuta y Melilla,0.0
4,1996,1,NaN,Libre,NaN,Ceuta y Melilla,Ceuta y Melilla,0.0


In [41]:
print("Regímenes:", df_tasa["Régimen"].unique())

print("\nComunidades autónomas:")
print(df_tasa["Comunidad_Autónoma"].unique())

print("\nCódigos territoriales:")
print(df_tasa["CODAUTO"].unique())

print("\nValores nulos:")
print(df_tasa.isna().sum())

print("\nPeriodo disponible:")
print(df_tasa["Año"].min(), df_tasa["Año"].max())

Regímenes: <ArrowStringArray>
['Libre', 'Protegida']
Length: 2, dtype: str

Comunidades autónomas:
<ArrowStringArray>
[            'Ceuta y Melilla',             'Castilla y León',
                   'Andalucía',                      'Aragón',
                     'Galicia',          'Castilla-La Mancha',
        'Comunitat Valenciana',                 'Extremadura',
              'Balears, Illes',                    'Canarias',
                   'Cantabria',                    'Cataluña',
                  'País Vasco',                       'Ceuta',
                     'Melilla',                   'Rioja, La',
 'Navarra, Comunidad Foral de',     'Asturias, Principado de',
        'Madrid, Comunidad de',                           nan,
           'Murcia, Region de',              'Total nacional']
Length: 22, dtype: str

Códigos territoriales:
[ 0.  7.  1.  2. 12.  8. 10. 11.  4.  5.  6.  9. 16. 18. 19. 17. 15.  3.
 13. nan]

Valores nulos:
Año                      0
Trimestre       

In [42]:
display(
    df_tasa[
        (df_tasa["Régimen"] == "Libre") &
        (
            df_tasa["Comunidad_Autónoma"].isin(
                ["Asturias, Principado de", "Total nacional"]
            )
        )
    ][
        ["Año", "Trimestre", "Valor", "Provincia",
         "Comunidad_Autónoma", "CPRO", "CODAUTO"]
    ].tail(20)
)

,Año,Trimestre,Valor,Provincia,Comunidad_Autónoma,CPRO,CODAUTO
13691,2007,2,2054.5,Total CCAA,Total nacional,NaN,NaN
13692,2008,2,2095.7,Total CCAA,Total nacional,NaN,NaN
13693,2009,2,1920.9,Total CCAA,Total nacional,NaN,NaN
13694,2010,2,1848.9,Total CCAA,Total nacional,NaN,NaN
13695,2011,2,1752.1,Total CCAA,Total nacional,NaN,NaN
13696,2012,2,1606.4,Total CCAA,Total nacional,NaN,NaN
13697,2013,2,1502.8,Total CCAA,Total nacional,NaN,NaN
13698,2014,2,1459.3,Total CCAA,Total nacional,NaN,NaN
13699,2015,2,1476.8,Total CCAA,Total nacional,NaN,NaN
13700,2016,2,1506.4,Total CCAA,Total nacional,NaN,NaN


In [43]:
print(
    df_tasa.groupby(
        ["Comunidad_Autónoma", "Régimen"],
        dropna=False
    ).size()
)

Comunidad_Autónoma           Régimen  
Andalucía                    Libre        1134
                             Protegida     774
Aragón                       Libre         504
                             Protegida     344
Asturias, Principado de      Libre         126
                             Protegida      86
Balears, Illes               Libre         126
                             Protegida      86
Canarias                     Libre         378
                             Protegida     258
Cantabria                    Libre         126
                             Protegida      86
Castilla y León              Libre        1260
                             Protegida     860
Castilla-La Mancha           Libre         756
                             Protegida     516
Cataluña                     Libre         630
                             Protegida     430
Ceuta                        Libre         126
                             Protegida      86
Ceuta y Melilla      

In [44]:
display(
    df_tasa[
        (df_tasa["Régimen"] == "Libre") &
        (df_tasa["Comunidad_Autónoma"].isin([
            "Asturias, Principado de",
            "Andalucía",
            "Total nacional"
        ]))
    ]
    .groupby(
        ["Comunidad_Autónoma", "Provincia", "CPRO"],
        dropna=False
    )
    .size()
    .reset_index(name="registros")
)

,Comunidad_Autónoma,Provincia,CPRO,registros
0,Andalucía,Almería,4.0,126
1,Andalucía,Cádiz,11.0,126
2,Andalucía,Córdoba,14.0,126
3,Andalucía,Granada,18.0,126
4,Andalucía,Huelva,21.0,126
5,Andalucía,Jaén,23.0,126
6,Andalucía,Málaga,29.0,126
7,Andalucía,Sevilla,41.0,126
8,Andalucía,Total CCAA,NaN,126
9,"Asturias, Principado de",Total CCAA,NaN,126


In [49]:
tasa = df_tasa[
    (df_tasa["Régimen"] == "Libre") &
    (
        (df_tasa["Provincia"] == "Total CCAA") |
        (df_tasa["Comunidad_Autónoma"].isin([
            "Balears, Illes",
            "Cantabria",
            "Rioja, La",
            "Ceuta",
            "Melilla"
        ]))
    )
].copy()

print("Dimensiones:", tasa.shape)
print("Territorios:", tasa["Comunidad_Autónoma"].nunique())

print("\nRegistros por territorio:")
print(tasa.groupby("Comunidad_Autónoma").size())

print("\nValores nulos:", tasa["Valor"].isna().sum())

print("\nDuplicados:", tasa.duplicated(
    subset=["Comunidad_Autónoma", "Año", "Trimestre"]
).sum())

Dimensiones: (2630, 8)
Territorios: 20

Registros por territorio:
Comunidad_Autónoma
Andalucía                      126
Aragón                         126
Asturias, Principado de        126
Balears, Illes                 126
Canarias                       126
Cantabria                      126
Castilla y León                126
Castilla-La Mancha             126
Cataluña                       126
Ceuta                          126
Comunitat Valenciana           126
Extremadura                    126
Galicia                        126
Madrid, Comunidad de           126
Melilla                        126
Murcia, Region de              126
Navarra, Comunidad Foral de    236
País Vasco                     126
Rioja, La                      126
Total nacional                 126
dtype: int64

Valores nulos: 72

Duplicados: 110


In [50]:
display(
    tasa[tasa["Comunidad_Autónoma"] == "Navarra, Comunidad Foral de"]
    .sort_values(["Año", "Trimestre"])
    .head(20)
)

,Año,Trimestre,Valor,Régimen,CPRO,Provincia,Comunidad_Autónoma,CODAUTO
10682,1995,1,722.8,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10683,1995,1,722.8,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10622,1995,2,742.7,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10623,1995,2,742.7,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10564,1995,3,759.6,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10565,1995,3,759.6,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10506,1995,4,721.3,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10507,1995,4,721.3,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10684,1996,1,736.6,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0
10685,1996,1,736.6,Libre,NaN,Total CCAA,"Navarra, Comunidad Foral de",15.0


In [51]:
territorios_ausentes = [
    "Balears, Illes",
    "Cantabria",
    "Rioja, La",
    "Ceuta",
    "Melilla"
]

display(
    df_tasa[
        (df_tasa["Régimen"] == "Libre") &
        (df_tasa["Comunidad_Autónoma"].isin(territorios_ausentes))
    ]
    .groupby(
        ["Comunidad_Autónoma", "Provincia", "CPRO"],
        dropna=False
    )
    .size()
    .reset_index(name="registros")
)

,Comunidad_Autónoma,Provincia,CPRO,registros
0,"Balears, Illes","Balears, Illes",7.0,126
1,Cantabria,Cantabria,39.0,126
2,Ceuta,Ceuta,51.0,126
3,Melilla,Melilla,52.0,126
4,"Rioja, La","Rioja, La",26.0,126


In [52]:
conflictos = (
    tasa.groupby(
        ["Comunidad_Autónoma", "Año", "Trimestre"]
    )["Valor"]
    .nunique()
)

print("Periodos con valores diferentes:", (conflictos > 1).sum())
print("Duplicados exactos:", tasa.duplicated().sum())
print("Territorios:", tasa["Comunidad_Autónoma"].nunique())

Periodos con valores diferentes: 0
Duplicados exactos: 110
Territorios: 20


In [53]:
tasa = tasa.drop_duplicates().copy()

tasa["territorio"] = tasa["Comunidad_Autónoma"].replace({
    "Total nacional": "Nacional",
    "Castilla-La Mancha": "Castilla - La Mancha",
    "Murcia, Region de": "Murcia, Región de"
})

tasa["cod_territorio"] = tasa["territorio"].map(
    mapa_territorios
)

print("Dimensiones:", tasa.shape)
print("Territorios:", tasa["territorio"].nunique())
print("Códigos nulos:", tasa["cod_territorio"].isna().sum())

print("Duplicados:", tasa.duplicated(
    subset=["territorio", "Año", "Trimestre"]
).sum())

display(
    tasa[["cod_territorio", "territorio"]]
    .drop_duplicates()
    .sort_values("cod_territorio", na_position="first")
)

Dimensiones: (2520, 10)
Territorios: 20
Códigos nulos: 126
Duplicados: 0


,cod_territorio,territorio
13585,NaN,Nacional
10742,01,Andalucía
10773,02,Aragón
10804,03,"Asturias, Principado de"
1434,04,"Balears, Illes"
10835,05,Canarias
1225,06,Cantabria
10866,07,Castilla y León
10897,08,Castilla - La Mancha
10928,09,Cataluña


In [60]:
tasa_limpia = tasa[
    ["cod_territorio", "territorio", "Año", "Trimestre", "Valor"]
].copy()

tasa_limpia = tasa_limpia.rename(columns={
    "Año": "anio",
    "Trimestre": "trimestre",
    "Valor": "valor_m2"
})

tasa_limpia = tasa_limpia.sort_values(
    ["cod_territorio", "anio", "trimestre"],
    na_position="first"
).reset_index(drop=True)

print("Nulos en tasa:", tasa["Valor"].isna().sum())
print("Nulos en tasa_limpia:", tasa_limpia["valor_m2"].isna().sum())

Nulos en tasa: 72
Nulos en tasa_limpia: 72


In [61]:
nulos_tasa = tasa_limpia[tasa_limpia["valor_m2"].isna()]

print("Nulos por territorio:")
print(nulos_tasa.groupby("territorio").size())

display(nulos_tasa.head(20))

Nulos por territorio:
territorio
Ceuta      36
Melilla    36
dtype: int64


,cod_territorio,territorio,anio,trimestre,valor_m2
2268,18,Ceuta,1995,1,NaN
2269,18,Ceuta,1995,2,NaN
2270,18,Ceuta,1995,3,NaN
2271,18,Ceuta,1995,4,NaN
2272,18,Ceuta,1996,1,NaN
2273,18,Ceuta,1996,2,NaN
2274,18,Ceuta,1996,3,NaN
2275,18,Ceuta,1996,4,NaN
2276,18,Ceuta,1997,1,NaN
2277,18,Ceuta,1997,2,NaN


In [64]:
display(
    tasa_limpia[tasa_limpia["valor_m2"].isna()]
    .groupby(["territorio", "anio"])
    .size()
)

territorio  anio
Ceuta       1995    4
            1996    4
            1997    4
            1998    4
            1999    4
            2000    4
            2001    4
            2002    4
            2003    4
Melilla     1995    4
            1996    4
            1997    4
            1998    4
            1999    4
            2000    4
            2001    4
            2002    4
            2003    4
dtype: int64

In [65]:
print("Nulos:", tasa_limpia["valor_m2"].isna().sum())

print("Tipo de dato:", tasa_limpia["valor_m2"].dtype)

print("Valores iguales o inferiores a cero:",
      (tasa_limpia["valor_m2"].dropna() <= 0).sum())

print("Nulos fuera de Ceuta y Melilla:",
      tasa_limpia.loc[
          tasa_limpia["valor_m2"].isna()
          & ~tasa_limpia["territorio"].isin(["Ceuta", "Melilla"])
      ].shape[0])

Nulos: 72
Tipo de dato: float64
Valores iguales o inferiores a cero: 1
Nulos fuera de Ceuta y Melilla: 0


In [66]:
display(
    tasa_limpia.loc[
        tasa_limpia["valor_m2"] <= 0
    ]
)

,cod_territorio,territorio,anio,trimestre,valor_m2
2468,19,Melilla,2013,3,0.0


In [68]:
tasa_limpia.loc[
    (tasa_limpia["territorio"] == "Melilla") &
    (tasa_limpia["anio"] == 2013) &
    (tasa_limpia["trimestre"] == 3) &
    (tasa_limpia["valor_m2"] == 0),
    "valor_m2"
] = float("nan")

assert len(tasa_limpia) == 2520
assert tasa_limpia["territorio"].nunique() == 20
assert tasa_limpia["valor_m2"].isna().sum() == 73
assert (tasa_limpia["valor_m2"].dropna() > 0).all()
assert tasa_limpia["trimestre"].between(1, 4).all()

assert tasa_limpia.duplicated(
    subset=["cod_territorio", "anio", "trimestre"]
).sum() == 0

assert tasa_limpia.loc[
    tasa_limpia["territorio"] != "Nacional",
    "cod_territorio"
].notna().all()

print("Validaciones superadas")

Validaciones superadas


In [69]:
ruta_salida = Path("../data/processed/core/tasa.csv")

ruta_salida.parent.mkdir(parents=True, exist_ok=True)

tasa_limpia.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig",
    na_rep=""
)

tasa_exportado = pd.read_csv(
    ruta_salida,
    dtype={"cod_territorio": "string"}
)

print("Dimensiones:", tasa_exportado.shape)
print("Valores nulos:", tasa_exportado["valor_m2"].isna().sum())

display(tasa_exportado.head())

Dimensiones: (2520, 5)
Valores nulos: 73


,cod_territorio,territorio,anio,trimestre,valor_m2
0,<NA>,Nacional,1995,1,670.8
1,<NA>,Nacional,1995,2,686.0
2,<NA>,Nacional,1995,3,682.6
3,<NA>,Nacional,1995,4,692.7
4,<NA>,Nacional,1996,1,686.0
